# DraftKings MLB Optimization Project
Samuel McDonald

3/12/2026

##Project Overview
This project builds optimization models to select the best DraftKings MLB lineup on 09-19-2025

Two models were created:
1. Problem 1: The Crystal Ball Model:
- This model gathers actual stats from 09-19-2025. It than converts every player who played on that day stats into the proper DraftKings MLB Classic scoring criteria. The goal is to achieve the maximum score using real life data from that day that follow all of the DraftKings competition's rules.

2. Problem 2: Trying to beat the average points per game model:
- In this problem, I did not have access to actual performance from that day. My goal was to beat the average participant's Average Points per Game model. At first I created a Naive model that decides the optimal lineup based on a player's Average Points per Game. I tried beating this model by implementing a strategy where players are boosted when they are playing bad teams.

## Data Collection

Two data sources that I used:

• DraftKings salary CSV  
• MLB Stats API

The DraftKings file provides player salaries, positions, and average fantasy
points per game. The MLB Stats API provides the game schedule and final boxscore
statistics required to compute the actual DraftKings points.

The API schedule endpoint is used to match each DraftKings player with the
correct MLB game ID (gamePk).

`load_and_setup` is the main function that gets everything ready before anything can happen. It takes two inputs: the date I care about which is the September 19, 2025 games, and the file path to the DraftKings salary CSV. It does these things in order: loads and cleans the DraftKings file, pulls the MLB schedule from the API for that date, pulls a team ID-to-abbreviation mapping from the API, merges all of that together so every player has a game ID attached to them, creates two flags on each player marking them as either a pitcher or a hitter, and finally packages everything into a dictionary that the Pyomo model can use directly. Inside of this function is a small helper called `pos_set`. This takes a position like "OF" or "2B" and returns a list of every player ID that fits that position. It works by scanning the Roster Position column and checking if that position is as a standalone word.

In [ ]:

# Mount Google Drive to access DraftKings CSV
from google.colab import drive
drive.mount('/content/drive')
# File paths are configured for local/Google Drive use
# Update paths as needed to run this project
import pandas as pd
import requests
import pyomo.environ as pe
import re
import unicodedata #I did this to strip accent marks and to get players in the same font


# Section 1: Load & Setup


def load_and_setup(date_str: str, dk_csv_path: str):
    #Step 1: Load and clean the DraftKings file
    dk = pd.read_csv(dk_csv_path)
    dk.columns = [c.strip() for c in dk.columns]
    dk = dk.rename(columns={"player_name": "Name"})

    keep = ["Name", "ID", "Roster Position", "Salary", "TeamAbbrev", "AvgPointsPerGame", "Game Info"]
    dk = dk[keep].copy()

    dk["Salary"] = pd.to_numeric(dk["Salary"], errors="coerce")
    dk["AvgPointsPerGame"] = pd.to_numeric(dk["AvgPointsPerGame"], errors="coerce")

    #Step 2: Pull MLB schedule for the selected date
    sched_url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date_str}"
    schedule = requests.get(sched_url).json()

    games = []
    for d in schedule.get("dates", []):
        for g in d.get("games", []):
            games.append({
                "gamePk":    g.get("gamePk"),
                "gameDate":  g.get("gameDate"),
                "home_id":   g["teams"]["home"]["team"]["id"],
                "home_name": g["teams"]["home"]["team"]["name"],
                "away_id":   g["teams"]["away"]["team"]["id"],
                "away_name": g["teams"]["away"]["team"]["name"],
                "status":    g.get("status", {}).get("detailedState")
            })
    games_df = pd.DataFrame(games)

    #Step 3: Pull MLB team abbreviation mapping:
    # DraftKings uses abbreviations (ex: LAD, BOS)
    # MLB schedule JSON uses numeric team IDs — I need this mapping to connect them
    teams_url = "https://statsapi.mlb.com/api/v1/teams?sportId=1"
    teams_json = requests.get(teams_url).json()

    teams_list = []
    for t in teams_json.get("teams", []):
        teams_list.append({
            "team_id":   t.get("id"),
            "abbrev":    t.get("abbreviation"),
            "team_name": t.get("name")
        })
    teams_df = pd.DataFrame(teams_list)

    #Step 4: Merge players with their game IDs
    home_map = games_df[["gamePk", "home_id"]].rename(columns={"home_id": "team_id"})
    away_map = games_df[["gamePk", "away_id"]].rename(columns={"away_id": "team_id"})
    team_game = pd.concat([home_map, away_map], ignore_index=True)
    team_game = team_game.merge(teams_df[["team_id", "abbrev"]], on="team_id", how="left")

    dk = dk.merge(
        team_game[["abbrev", "gamePk"]].drop_duplicates(),
        left_on="TeamAbbrev",
        right_on="abbrev",
        how="left"
    ).drop(columns=["abbrev"])

    #Step 5: Create helper flags for position constraints
    dk["is_pitcher"] = dk["Roster Position"].astype(str).str.contains(r"\bP\b", regex=True)
    dk["is_hitter"]  = ~dk["is_pitcher"]

    #Step 6: Build sets and parameter dictionaries for Pyomo
    players = dk["ID"].tolist()

    salary = dict(zip(dk["ID"], dk["Salary"]))
    proj   = dict(zip(dk["ID"], dk["AvgPointsPerGame"]))
    team   = dict(zip(dk["ID"], dk["TeamAbbrev"]))
    gamePk = dict(zip(dk["ID"], dk["gamePk"]))

    def pos_set(pos):
        # Allows multi-position players (ex: 1B/3B) to fill either slot
        return dk.loc[dk["Roster Position"].astype(str).str.contains(fr"\b{pos}\b", regex=True), "ID"].tolist()

    games_in_slate = sorted(dk["gamePk"].dropna().unique().tolist())
    teams_in_slate = sorted(dk["TeamAbbrev"].dropna().unique().tolist())

    data = {
        "players": players,
        "teams":   teams_in_slate,
        "games":   games_in_slate,

        # Position sets
        "P":  pos_set("P"),
        "C":  pos_set("C"),
        "1B": pos_set("1B"),
        "2B": pos_set("2B"),
        "3B": pos_set("3B"),
        "SS": pos_set("SS"),
        "OF": pos_set("OF"),

        #Parameters
        "salary": salary,
        "proj":   proj,
        "team":   team,
        "gamePk": gamePk,

        #Grouping helpers for constraints
        "players_by_game": {g: dk.loc[dk["gamePk"] == g, "ID"].tolist() for g in games_in_slate},
        "hitters_by_team": {t: dk.loc[(dk["TeamAbbrev"] == t) & (dk["is_hitter"]), "ID"].tolist()
                            for t in teams_in_slate},
    }

    return dk, games_df, data


#Run the setup
date_str    = "2025-09-19"
dk_csv_path = "/content/drive/MyDrive/DU 2025-2026/Winter 2026/INFO 3440 Project/DKSalaries0919.csv"

dk, games_df, data = load_and_setup(date_str, dk_csv_path)
dk.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Name,ID,Roster Position,Salary,TeamAbbrev,AvgPointsPerGame,Game Info,gamePk,is_pitcher,is_hitter
0,Zack Wheeler,40107950,P,10800,PHI,24.60,PHI@ARI 09/19/2025 09:40PM ET,776255.0,True,False
1,Garrett Crochet,40107348,P,10500,BOS,24.84,BOS@TB 09/19/2025 07:35PM ET,776261.0,True,False
2,Blake Snell,40107951,P,10300,LAD,20.59,SF@LAD 09/19/2025 10:10PM ET,776256.0,True,False
3,Yoshinobu Yamamoto,40107952,P,10300,LAD,21.26,SF@LAD 09/19/2025 10:10PM ET,776256.0,True,False
4,Nathan Eovaldi,40107953,P,10200,TEX,21.76,MIA@TEX 09/19/2025 08:05PM ET,776251.0,True,False


## Data Loading Results
The setup function above loads the DraftKings salary CSV and merges it with the MLB schedule which was pulled from the MLB Stats API for September 19, 2025. Each player was matched to their game using `gamePk` which is a unique way to identify each game that happened on that day. `gamePk` is important for my optimization model because it is used to enforce the rule that players come from at least two different games, and to track how many hitters I am taking from each team.

## Data Preparation

DraftKings and the MLB API sometimes use different team abbreviations.
For example:

WAS - WSH  
CHW - CWS  

A normalization step is implemented so that both datasets use the same
abbreviations. This allows the DraftKings players to be correctly matched
with their MLB games.

I made a small utility function called `fix_abbrev` that takes a team abbreviation and checks it against a lookup table of known mismatches between DraftKings and the MLB API. If the abbreviation is in the table, it returns the corrected version. If not, it returns it unchanged. For example, passing in "WAS" returns "WSH", and passing in "LAD" returns "LAD" since that one already matches. Every team abbreviation in the project runs through this function before any merging happens.

In [ ]:
# Section 2: Fix gamePk Mapping (abbreviation normalization)
# ============================================================
# DraftKings and the MLB API sometimes use slightly different team abbreviations
# (ex: DK uses "WAS" but MLB API uses "WSH"). This block recombines them.
# I do one merge here.

# Pull fresh team abbreviations from MLB API (id to abbreviation)
teams_url = "https://statsapi.mlb.com/api/v1/teams?sportId=1"
teams_json = requests.get(teams_url).json()
team_id_to_abbrev = {t["id"]: t["abbreviation"] for t in teams_json["teams"]}

# Add MLB abbreviations to the schedule dataframe
games_df["home_abbrev_raw"] = games_df["home_id"].map(team_id_to_abbrev)
games_df["away_abbrev_raw"] = games_df["away_id"].map(team_id_to_abbrev)

# Build matchup key from DK Game Info (first token like "PHI@ARI")
dk["matchup_raw"] = dk["Game Info"].astype(str).str.split().str[0].str.strip()
dk["away_raw"]    = dk["matchup_raw"].str.split("@").str[0].str.strip()
dk["home_raw"]    = dk["matchup_raw"].str.split("@").str[1].str.strip()

# Comprehensive alias map: DK abbreviation to MLB API abbreviation
# DK tends to use older/shorter forms; MLB API uses the official current ones
abbrev_fix = {
    # Washington Nationals
    "WAS": "WSH",
    # Chicago White Sox
    "CHW": "CWS",
    # Arizona Diamondbacks
    "AZ":  "ARI",
    # Los Angeles Dodgers short form
    "LA":  "LAD",
    # Tampa Bay Rays variants
    "TAM": "TB",
    "TBR": "TB",
    # San Diego Padres
    "SDP": "SD",
    # San Francisco Giants
    "SFG": "SF",
    # Cleveland Guardians (MLB API uses "CLE", DK sometimes uses "CLV")
    "CLV": "CLE",
    # Kansas City Royals (sometimes "KCR")
    "KCR": "KC",
    # Athletics (sometimes "OAK" or "ATH")
    "OAK": "ATH",
    # Pittsburgh Pirates (sometimes "PIT" is fine, but "PIT" = "PIT")
    # Miami Marlins (sometimes "FLA" in older data)
    "FLA": "MIA",
    # Toronto Blue Jays (sometimes "TOR" is fine)
    # Minnesota Twins (sometimes "MIN" is fine)
}

def fix_abbrev(a):
    if pd.isna(a):
        return None
    a = str(a).strip()
    return abbrev_fix.get(a, a)

# Make the same abbreviations on both DK and schedule sides
dk["away_fix"]       = dk["away_raw"].apply(fix_abbrev)
dk["home_fix"]       = dk["home_raw"].apply(fix_abbrev)
games_df["away_fix"] = games_df["away_abbrev_raw"].apply(fix_abbrev)
games_df["home_fix"] = games_df["home_abbrev_raw"].apply(fix_abbrev)

# Build normalized matchup keys (away@home) for both sides
dk["matchup"]       = dk["away_fix"] + "@" + dk["home_fix"]
games_df["matchup"] = games_df["away_fix"] + "@" + games_df["home_fix"]

# Single clean merge — drop any stale gamePk from load_and_setup first
dk = dk.drop(columns=["gamePk"], errors="ignore")
dk = dk.merge(
    games_df[["matchup", "gamePk"]].drop_duplicates(),
    on="matchup",
    how="left"
)

#Drop duplicate player rows that a many-to-one merge can introduce
dk = dk.drop_duplicates(subset=["ID"]).reset_index(drop=True)
missing_count = dk["gamePk"].isna().sum()

# If any are still missing, use a fallback: assign gamePk by TeamAbbrev alone
if missing_count > 0:
    # Fallback: assign gamePk by TeamAbbrev alone (each team plays exactly one game per day)
    team_to_gpk = {}
    for _, row in games_df.iterrows():
        if pd.notna(row.get("home_abbrev_raw")):
            team_to_gpk[fix_abbrev(row["home_abbrev_raw"])] = row["gamePk"]
        if pd.notna(row.get("away_abbrev_raw")):
            team_to_gpk[fix_abbrev(row["away_abbrev_raw"])] = row["gamePk"]

    def fallback_gpk(r):
        """
        This is a safety net function that runs if some players are still missing a gameID after my main merge.
        For any player without a game ID, it looks up their team abbreviation in a backup dictionary that maps every team to their game
        that day.
        """
        if pd.notna(r["gamePk"]):
            return r["gamePk"]
        return team_to_gpk.get(fix_abbrev(str(r["TeamAbbrev"]).strip()), float("nan"))

    dk["gamePk"] = dk.apply(fallback_gpk, axis=1)

# Rebuild all data-dict fields that depend on gamePk from the corrected dk.
# load_and_setup froze these from a partial gamePk column — we must overwrite them.
data["players"]         = dk["ID"].tolist()
data["salary"]          = dict(zip(dk["ID"], dk["Salary"]))
data["proj"]            = dict(zip(dk["ID"], dk["AvgPointsPerGame"]))
data["team"]            = dict(zip(dk["ID"], dk["TeamAbbrev"]))
data["gamePk"]          = dict(zip(dk["ID"], dk["gamePk"]))
data["games"]           = sorted(dk["gamePk"].dropna().unique().tolist())
data["players_by_game"] = {g: dk.loc[dk["gamePk"] == g, "ID"].tolist() for g in data["games"]}
data["hitters_by_team"] = {
    t: dk.loc[(dk["TeamAbbrev"] == t) & (dk["is_hitter"]), "ID"].tolist()
    for t in data["teams"]
}


## DraftKings Scoring Rules

Actual fantasy points are calculated using official DraftKings MLB scoring.

Hitter scoring includes:

• Single = 3 pts  
• Double = 5 pts  
• Triple = 8 pts  
• Home Run = 10 pts  
• RBI = 2 pts  
• Run = 2 pts  
• Walk = 2 pts  
• Stolen Base = 5 pts  

Pitcher scoring includes:

• Inning Pitched = 2.25 pts  
• Strikeout = 2 pts  
• Win = 4 pts  
• Earned Run = −2 pts  
• Hit Allowed = −0.6 pts  
• Walk = −0.6 pts
• Complete Game: +2.5
• Complete Game Shutout (CG + 0 runs allowed): +2.5 additional
• No-Hitter (CG + 0 hits allowed): +5.0 additional

## Scoring Helper Functions

I need four helper functions, before I can calculate actual DraftKings points.

Here is what each one does:

**normalize_name** — Cleans up player names so they match between DraftKings and
the MLB API. It lowercases everything, strips accent marks (so José becomes Jose),
removes punctuation, and drops suffixes like Jr. or III. Without this, Latin players
with accented names would score zero because their names would never match.

**ip_to_float** — Converts the MLB API's innings pitched format into a real number.
The MLB API writes innings pitched in a non-standard way: "5.2" means 5 and 2/3
innings, not 5.2 innings. This function converts that format correctly so pitcher
points are calculated accurately.

**dk_hitter_points** — Takes a hitter's box score stats and returns their DraftKings
fantasy score. The MLB API does not report singles directly, so the function calculates
them as: singles = hits − doubles − triples − home runs.

**dk_pitcher_points** — Takes a pitcher's box score stats and returns their DraftKings
fantasy score, including a win bonus if they were the winning pitcher that game.

**pitcher_bonuses_from_live** — Checks whether a pitcher earned bonus points for a
complete game, complete game shutout, or no-hitter. These bonuses require checking
the live game feed rather than just the boxscore, because the live feed is the only
place that tells us how many pitchers a team used.

In [ ]:

# Section 3: Scoring Helper Functions
# ============================================================

def normalize_name(name):
    if pd.isna(name):
        return ""
    s = str(name).lower().strip()
    # Strip accent marks (ex: é -> e, á -> a)
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    s = re.sub(r"[,.\-']", " ", s)
    s = re.sub(r"\b(jr|sr|ii|iii|iv|v)\b", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def ip_to_float(ip_str):
#Converts MLB innings pitched format (ex: '5.2' = 5 2/3 innings) to a float.
#MLB uses .1 for 1/3 inning and .2 for 2/3 inning — NOT decimal notation.

    if ip_str is None:
        return 0.0
    s = str(ip_str).strip()
    if s == "" or s.lower() == "nan":
        return 0.0
    if "." not in s:
        return float(s)
    whole, frac = s.split(".")
    whole = int(whole) if whole else 0
    frac  = int(frac)  if frac  else 0
    if frac == 0:
        return float(whole)
    elif frac == 1:
        return whole + (1 / 3)
    elif frac == 2:
        return whole + (2 / 3)
    else:
        return float(whole)


def dk_hitter_points(b):
#MLB API doesn't provide singles directly, so singles = hits - 2B - 3B - HR

    hits    = b.get("hits", 0) or 0
    doubles = b.get("doubles", 0) or 0
    triples = b.get("triples", 0) or 0
    hrs     = b.get("homeRuns", 0) or 0
    singles = max(hits - doubles - triples - hrs, 0)

    rbi = b.get("rbi", 0) or 0
    runs = b.get("runs", 0) or 0
    bb  = b.get("baseOnBalls", 0) or 0
    sb  = b.get("stolenBases", 0) or 0
    hbp = b.get("hitByPitch", 0) or 0

    pts  = 3  * singles
    pts += 5  * doubles
    pts += 8  * triples
    pts += 10 * hrs
    pts += 2  * rbi
    pts += 2  * runs
    pts += 2  * bb
    pts += 5  * sb
    pts += 2  * hbp
    return pts


def dk_pitcher_points(p, is_win=False):
#Computes DraftKings fantasy points for a pitcher using MLB API pitching stats.

    ip          = ip_to_float(p.get("inningsPitched", "0.0"))
    strikeouts  = p.get("strikeOuts", 0) or 0
    er          = p.get("earnedRuns", 0) or 0
    h           = p.get("hits", 0) or 0
    bb          = p.get("baseOnBalls", 0) or 0
    hbp_against = p.get("hitByPitch", 0) or 0

    pts  = 2.25 * ip
    pts += 2.0  * strikeouts
    pts += 4.0  * (1 if is_win else 0)


    pts += -2.0 * er
    pts += -0.6 * h
    pts += -0.6 * bb
    pts += -0.6 * hbp_against
    return pts


def pitcher_bonuses_from_live(live, side_key, pid):
  #Uses the live feed (which has decisions/pitchers_used) for accuracy.
    bonus = 0.0
    teams_box   = live.get("liveData", {}).get("boxscore", {}).get("teams", {})
    this_side   = teams_box.get(side_key, {})
    other_side  = teams_box.get("away" if side_key == "home" else "home", {})

    other_batting_totals = other_side.get("teamStats", {}).get("batting", {}) or {}
    hits_against_team    = other_batting_totals.get("hits", 0) or 0
    runs_against_team    = other_batting_totals.get("runs", 0) or 0

    pitchers_used = this_side.get("pitchers", []) or []
    starters      = this_side.get("startingPitcher", {}) or {}
    starter_id    = starters.get("id", None)

    # Complete Game: team used exactly 1 pitcher, who is the starter and is our pitcher
    if len(pitchers_used) == 1 and starter_id is not None:
        if int(pitchers_used[0]) == int(starter_id) == int(pid):
            bonus += 2.5  # Complete Game

            if runs_against_team == 0:
                bonus += 2.5  # Complete Game Shutout

            if hits_against_team == 0:
                bonus += 5.0  # No-Hitter

    return bonus

## Calculating Actual points from MLB API.
This section calculates actual points from the MLB API and converts them into DraftKings fantasy points for every player who played on September 19, 2025.

First, I normalizeevery player name in the DraftKings file using the `normalize_name` function so they are ready to be matched against the MLB API later. This helps all player names be consistent.

Then I filter the schedule down to only games that were actually completed that day, postponed or cancelled games are ignored and stay at zero points.

Next, I create a blank dictionary called `actual_lookup`. This dictionary will store each player's DraftKings fantasy score, labeled by their normalized name, their team abbreviation, and the game ID. Using all three as the determents prevents any chance of two
players from different games accidentally overwriting each other.

I then use nested for loops to go through every completed game. For each game, I pull two things from the MLB API: the boxscore (which has every player's individual stats) and the live feed (which tells me who the winning pitcher was and whether any pitcher
threw a complete game). I loop through both the home and away side of each game, check whether each player pitched or hit, and call the appropriate scoring function to calculate their DraftKings points. Each player's score gets saved into `actual_lookup`
under both their raw MLB abbreviation and their normalized DraftKings abbreviation, so the lookup works no matter which format gets used later.

Finally, the `safe_lookup_actual` function attaches each player's actual score back to the main DraftKings dataframe. Team abbreviations sometimes differ between DraftKings and the MLB API, it tries three different abbreviation formats in order until it finds a match. If none of the three work, the player's score stays at zero.
Once every player has their actual score attached, the results are stored in the data dictionary so the Pyomo optimization model can use them.

In [ ]:

# Section 4: Calculate Actual Points from MLB API
# I only calculate actual points for games that were actually completed.
# Postponed/cancelled games stay at 0.

dk["Name_norm"] = dk["Name"].apply(normalize_name)

played_games = games_df[games_df["status"].isin(["Final", "Game Over"])].copy()
games_played = played_games["gamePk"].tolist()

# Lookup dictionary: (normalized_name, team_abbrev, gamePk) to DK fantasy points
# Using gamePk as a third key prevents accidental cross-game overwrites
actual_lookup = {}

for g in games_played:

    # Use boxscore for player stats (recommended endpoint)
    box_url = f"https://statsapi.mlb.com/api/v1/game/{int(g)}/boxscore"
    box = requests.get(box_url).json()

    # Use live feed only to identify the winning pitcher and pitcher bonuses
    live_url = f"https://statsapi.mlb.com/api/v1.1/game/{int(g)}/feed/live"
    live = requests.get(live_url).json()

    # Find the winning pitcher's ID (needed for the Win bonus)
    winner_id   = None
    win_pitcher = live.get("liveData", {}).get("decisions", {}).get("winner", {})
    if isinstance(win_pitcher, dict) and "id" in win_pitcher:
        winner_id = int(win_pitcher["id"])

    # Loop through both home and away sides
    for side_key in ["home", "away"]:

        # Get team abbreviation from boxscore (used to match DK players)
        team_abbrev = box.get("teams", {}).get(side_key, {}).get("team", {}).get("abbreviation", None)

        side         = box.get("teams", {}).get(side_key, {})
        players_dict = side.get("players", {})

        for _, pobj in players_dict.items():
            person    = pobj.get("person", {})
            full_name = person.get("fullName", "")
            name_norm = normalize_name(full_name)

            batting  = pobj.get("stats", {}).get("batting", {})  or {}
            pitching = pobj.get("stats", {}).get("pitching", {}) or {}

            pitched_ip = ip_to_float(pitching.get("inningsPitched", "0.0"))
            is_pitcher = pitched_ip > 0

            if is_pitcher:
                pid    = person.get("id", None)
                is_win = (winner_id is not None and pid is not None and int(pid) == winner_id)
                pts    = dk_pitcher_points(pitching, is_win=is_win)
                if pid is not None:
                    pts += pitcher_bonuses_from_live(live, side_key, pid)
            else:
                pts = dk_hitter_points(batting)

            if team_abbrev is not None:
                # Store under both the raw MLB abbreviation AND the DK-normalized version.
                # This ensures the lookup succeeds even when MLB API returns ex: "KCR" but DK uses "KC".
                actual_lookup[(name_norm, team_abbrev, int(g))]             = pts
                actual_lookup[(name_norm, fix_abbrev(team_abbrev), int(g))] = pts


# Build a reverse map from DK abbreviation to MLB API raw abbreviation.
# ex: "KC" to "KCR" if the boxscore returns "KCR".
dk_to_mlb_abbrev = {}
for _, row in games_df.iterrows():
    for raw_col in ["home_abbrev_raw", "away_abbrev_raw"]:
        mlb_a = row.get(raw_col)
        if pd.notna(mlb_a):
            dk_to_mlb_abbrev[fix_abbrev(str(mlb_a))] = str(mlb_a)

# Attach actual points back to the DraftKings dataframe.
# Tries multiple abbreviation variants so a mismatch doesn't silently zero out a player.
def safe_lookup_actual(r):
    if pd.isna(r["gamePk"]):
        return 0.0
    g       = int(r["gamePk"])
    nn      = r["Name_norm"]
    dk_abbr = r["TeamAbbrev"]

    # Try 1: exact DK abbreviation as-is
    pts = actual_lookup.get((nn, dk_abbr, g), None)
    if pts is not None:
        return pts

    # Try 2: fix_abbrev-normalized DK abbreviation (ex. "WAS" - "WSH")
    pts = actual_lookup.get((nn, fix_abbrev(dk_abbr), g), None)
    if pts is not None:
        return pts

    # Try 3: MLB API raw abbreviation (ex. DK "KC" - MLB "KCR")
    mlb_raw = dk_to_mlb_abbrev.get(fix_abbrev(dk_abbr))
    if mlb_raw:
        pts = actual_lookup.get((nn, mlb_raw, g), None)
        if pts is not None:
            return pts

    return 0.0

dk["ActualPoints"] = dk.apply(safe_lookup_actual, axis=1)

# Store in data dict for Pyomo
data["actual"] = dict(zip(dk["ID"], dk["ActualPoints"]))

print("\nTop 10 Actual DK Scorers:")
display(dk[["Name", "TeamAbbrev", "Roster Position", "Salary", "ActualPoints"]]
        .sort_values("ActualPoints", ascending=False).head(10))


Top 10 Actual DK Scorers:


,Name,TeamAbbrev,Roster Position,Salary,ActualPoints
1035,Michael Massey,KC,2B/OF,2100,33.00
213,Juan Soto,NYM,OF,6200,32.00
23,Trevor Rogers,BAL,P,8900,29.10
1002,Jac Caglianone,KC,OF,2400,29.00
5,Bryan Woo,SEA,P,10000,28.05
344,Yandy Diaz,TB,1B,4700,26.00
1,Garrett Crochet,BOS,P,10500,25.30
314,CJ Abrams,WSH,SS,5100,25.00
32,Sonny Gray,STL,P,8700,24.10
137,Michael Lorenzen,KC,P,6700,23.65


## Problem 1: The Crystal Ball Optimization
Now that I have every player's actual DraftKings score from September 19, 2025,
I can try finding the best possible lineup someone could have picked that day.

This is called a crystal ball model, it uses hindsight to find the
mathematical ceiling. No real contest player could do this, but it gives us a benchmark to measure every other strategy against.

To find this lineup, I use an integer linear program built with Pyomo. Each player
gets a binary decision variable: 1 if selected, 0 if not. The solver maximizes total
actual DraftKings points while satisfying every DraftKings roster rule:

- Total salary cannot exceed $50,000
- Exactly 10 players must be selected
- Roster must include 2 pitchers, 1 catcher, 1 first baseman, 1 second baseman,
  1 third baseman, 1 shortstop, and 3 outfielders
- No more than 5 hitters from any one team
- Players must come from at least 2 different games

Two details worth noting: Michael Massey is listed as 2B/OF, meaning he qualifies
for both positions. To handle this correctly, I use minimum thresholds (>=) for
each position rather than strict equality, paired with upper bound caps. The exact
10-player constraint then handles the final count. Another problem is players with accents in their name such as José Ramírez, wouldn't have their accents stripped which would cause the model to not factor them into the ideal lineup. This was fixed by using unicodedata to strip accent marks and re to strip suffixes.

In [ ]:
# PROBLEM 1 — Crystal Ball Optimization
# Optimal lineup if you KNEW how every player would perform that day.
# Objective: Maximize actual DraftKings points earned on 9/19/2025.
# ============================================================

DV_indexes = data["players"]
games      = data["games"]
teams      = data["teams"]

model_1 = pe.ConcreteModel()

# Decision variables:
# x[i] = 1 if player i is selected, 0 otherwise
# y[g] = 1 if at least one player from game g is selected
model_1.x = pe.Var(DV_indexes, domain=pe.Binary)
model_1.y = pe.Var(games, domain=pe.Binary)

# Objective: Maximize total actual fantasy points
model_1.obj = pe.Objective(
    expr=sum(data["actual"][i] * model_1.x[i] for i in DV_indexes),
    sense=pe.maximize
)

#Constraints

# Total salary must not exceed $50,000
model_1.salary_cap = pe.Constraint(
    expr=sum(data["salary"][i] * model_1.x[i] for i in DV_indexes) <= 50000
)

# Exactly 10 players on the roster
model_1.total_players = pe.Constraint(
    expr=sum(model_1.x[i] for i in DV_indexes) == 10
)

# Position requirements (DraftKings MLB Classic roster)
# I use >= (not ==) because multi-position players (ex: "2B/OF") are counted
# in multiple position sets simultaneously. A player like Massey (2B/OF) counts
# toward both the 2B sum and the OF sum when selected. Using == would make the
# model infeasible: selecting Massey satisfies 2B>=1 AND contributes to OF>=3,
# so the total_players==10 constraint correctly handles the exact count.
model_1.twoP    = pe.Constraint(expr=sum(model_1.x[i] for i in data["P"])  >= 2)
model_1.oneC    = pe.Constraint(expr=sum(model_1.x[i] for i in data["C"])  >= 1)
model_1.one1B   = pe.Constraint(expr=sum(model_1.x[i] for i in data["1B"]) >= 1)
model_1.one2B   = pe.Constraint(expr=sum(model_1.x[i] for i in data["2B"]) >= 1)
model_1.one3B   = pe.Constraint(expr=sum(model_1.x[i] for i in data["3B"]) >= 1)
model_1.oneSS   = pe.Constraint(expr=sum(model_1.x[i] for i in data["SS"]) >= 1)
model_1.threeOF = pe.Constraint(expr=sum(model_1.x[i] for i in data["OF"]) >= 3)

# Upper bound caps prevent the solver from stacking a single position.
# Bounds are intentionally loose to accommodate multi-position players:
# ex: Massey (2B/OF) counts toward both 2B and OF sets simultaneously,
# so capping OF at 3 would incorrectly block him from filling the OF slot.
# total_players==10 combined with the >= minimums enforces the correct roster.
model_1.maxP  = pe.Constraint(expr=sum(model_1.x[i] for i in data["P"])  <= 2)
model_1.maxC  = pe.Constraint(expr=sum(model_1.x[i] for i in data["C"])  <= 1)
model_1.max1B = pe.Constraint(expr=sum(model_1.x[i] for i in data["1B"]) <= 2)
model_1.max2B = pe.Constraint(expr=sum(model_1.x[i] for i in data["2B"]) <= 2)
model_1.max3B = pe.Constraint(expr=sum(model_1.x[i] for i in data["3B"]) <= 2)
model_1.maxSS = pe.Constraint(expr=sum(model_1.x[i] for i in data["SS"]) <= 1)
model_1.maxOF = pe.Constraint(expr=sum(model_1.x[i] for i in data["OF"]) <= 4)

# Max 5 hitters (non-pitchers) from any single team
model_1.max_hitters = pe.ConstraintList()
for t in teams:
    model_1.max_hitters.add(
        sum(model_1.x[i] for i in data["hitters_by_team"][t]) <= 5
    )

# At least 2 different games must be represented
# Big-M linking constraint: if any player from game g is selected, y[g] = 1
model_1.link_game = pe.ConstraintList()
for g in games:
    model_1.link_game.add(
        sum(model_1.x[i] for i in data["players_by_game"][g]) <= 10 * model_1.y[g]
    )
model_1.two_games = pe.Constraint(
    expr=sum(model_1.y[g] for g in games) >= 2
)

# Solve
opt = pe.SolverFactory("appsi_highs")
opt.solve(model_1)

# Extract and display the optimal lineup
chosen_id_1 = [i for i in DV_indexes if pe.value(model_1.x[i]) > 0.5]
lineup1 = dk[dk["ID"].isin(chosen_id_1)].copy()
lineup1 = lineup1[[
    "Name", "TeamAbbrev", "Roster Position", "Salary",
    "ActualPoints", "AvgPointsPerGame", "Game Info", "gamePk"
]].sort_values(by="ActualPoints", ascending=False)

total_salary = lineup1["Salary"].sum()
total_points = lineup1["ActualPoints"].sum()


## Problem 1 Results

The crystal ball model found the most optimal lineup for September 19,2025. This is the highest score achievable under all DraftKings rules. This is my ceiling for the rest of the project. My strategy will be compared with 268.15 which was the max of the total actual points that day for an optimal lineup.

In [ ]:
print("PROBLEM 1 RESULTS: Crystal Ball Optimization")
print("=" * 50)
print(f"Total Salary Used:    ${total_salary:,.0f}")
print(f"Total ACTUAL Points:  {total_points:.2f}")
display(lineup1)

# Constraint verification
print("\n--- Constraint Checks ---")
print("Players selected:", len(lineup1))
print("Pitchers selected:", lineup1["Roster Position"].str.contains(r"\bP\b", regex=True).sum())
print("Unique games represented:", lineup1["gamePk"].nunique())
hitters_only = lineup1[~lineup1["Roster Position"].str.contains(r"\bP\b", regex=True)]
print("Max hitters from one team:", hitters_only.groupby("TeamAbbrev")["Name"].count().max())

PROBLEM 1 RESULTS: Crystal Ball Optimization
Total Salary Used:    $49,300
Total ACTUAL Points:  268.15


,Name,TeamAbbrev,Roster Position,Salary,ActualPoints,AvgPointsPerGame,Game Info,gamePk
1035,Michael Massey,KC,2B/OF,2100,33.00,3.75,TOR@KC 09/19/2025 07:40PM ET,776260
213,Juan Soto,NYM,OF,6200,32.00,10.58,WSH@NYM 09/19/2025 07:10PM ET,776259
23,Trevor Rogers,BAL,P,8900,29.10,22.52,NYY@BAL 09/19/2025 07:05PM ET,776273
1002,Jac Caglianone,KC,OF,2400,29.00,4.42,TOR@KC 09/19/2025 07:40PM ET,776260
5,Bryan Woo,SEA,P,10000,28.05,21.53,SEA@HOU 09/19/2025 08:10PM ET,776262
344,Yandy Diaz,TB,1B,4700,26.00,8.20,BOS@TB 09/19/2025 07:35PM ET,776261
314,CJ Abrams,WSH,SS,5100,25.00,8.64,WSH@NYM 09/19/2025 07:10PM ET,776259
891,Carter Jensen,KC,C,3100,23.00,7.09,TOR@KC 09/19/2025 07:40PM ET,776260
407,Harrison Bader,PHI,OF,4000,22.00,6.46,PHI@ARI 09/19/2025 09:40PM ET,776255
946,Miguel Rojas,LAD,2B/3B,2800,21.00,4.40,SF@LAD 09/19/2025 10:10PM ET,776256



--- Constraint Checks ---
Players selected: 10
Pitchers selected: 2
Unique games represented: 7
Max hitters from one team: 3


## Results

Total Salary Used: $49,300  
Total Actual Points: 268.15  

The model generated an optimal lineup while satisfying all constraints, outperforming average-based player selection strategies.

## Strategy: Target Weak Opponents

Instead of selecting players purely based on their historical average points,
this model adjusts projections based on the strength of the opponent.

Teams with a win percentage below .450 were classified as weak teams.

Players facing these teams received a projection boost:

• Hitters: +15%
• Pitchers: +10%

The optimization model then maximizes the adjusted projection rather than
raw average points.

I compare MY picking on the weak teams model to:
  1. Crystal Ball lineup (Problem 1 ceiling)
  2. Naive lineup using raw AvgPointsPerGame
  3. Strategy lineup using AdjustedProj

In [ ]:
# PROBLEM 2 — Weak Opponent Strategy
# ============================================================

# Step 1: Pull MLB standings as of 2025-09-19 from the MLB Stats API
# and identify weak teams (win percentage < .450).
# The standings endpoint returns cumulative records up to the date provided.


standings_url = "https://statsapi.mlb.com/api/v1/standings?leagueId=103,104&season=2025&date=2025-09-19&standingsTypes=regularSeason"
standings_json = requests.get(standings_url).json()

team_records = []
for division in standings_json.get("records", []):
    for entry in division.get("teamRecords", []):
        team_id   = entry["team"]["id"]
        team_name = entry["team"]["name"]
        wins      = entry["wins"]
        losses    = entry["losses"]
        win_pct   = wins / (wins + losses) if (wins + losses) > 0 else 0.0
        abbrev    = team_id_to_abbrev.get(team_id, "???")
        team_records.append({
            "team_id":   team_id,
            "team_name": team_name,
            "abbrev":    abbrev,
            "abbrev_fix": fix_abbrev(abbrev),
            "wins":      wins,
            "losses":    losses,
            "win_pct":   win_pct,
        })

standings_df = pd.DataFrame(team_records).sort_values("win_pct")

print("MLB Standings as of 2025-09-19 (sorted by win%):")
# Weak teams: win percentage strictly below .450
weak_teams = set(standings_df.loc[standings_df["win_pct"] < 0.450, "abbrev_fix"].tolist())

display(standings_df[standings_df["abbrev_fix"].isin(weak_teams)][["team_name", "abbrev_fix", "wins", "losses", "win_pct"]].head(10))

print("\nWeak teams (win pct < .450) (Top 10):", sorted(weak_teams)[:10])

MLB Standings as of 2025-09-19 (sorted by win%):


,team_name,abbrev_fix,wins,losses,win_pct
29,Rockies,COL,42,112,0.272727
9,White Sox,CWS,58,96,0.376623
19,Nationals,WSH,62,92,0.402597
24,Pirates,PIT,65,89,0.422078
8,Twins,MIN,66,87,0.431373
14,Angels,LAA,69,85,0.448052



Weak teams (win pct < .450) (Top 10): ['COL', 'CWS', 'LAA', 'MIN', 'PIT', 'WSH']


## Problem 2: Building the Weak Opponent Strategy
With the weak teams identified from the MLB Standings, I now need to figure out which players are actually playing those teams on 09-19-2025. Each players opponent is from the DraftKings CSV file. Once I know each player's opponent, I apply the projection boost and rebuild the optimization model using adjusted scores instead of raw Average Points per Game.

In [ ]:
# Step 2: Identify opponent for each player
# ================================================================================
# TeamFix, away_fix, and home_fix should already exist from earlier matchup work.
# If not, this rebuilds them safely.

dk["TeamFix"] = dk["TeamAbbrev"].apply(fix_abbrev)

if "matchup_raw" not in dk.columns:
    dk["matchup_raw"] = dk["Game Info"].astype(str).str.split().str[0].str.strip()

if "away_raw" not in dk.columns:
    dk["away_raw"] = dk["matchup_raw"].str.split("@").str[0].str.strip()

if "home_raw" not in dk.columns:
    dk["home_raw"] = dk["matchup_raw"].str.split("@").str[1].str.strip()

if "away_fix" not in dk.columns:
    dk["away_fix"] = dk["away_raw"].apply(fix_abbrev)

if "home_fix" not in dk.columns:
    dk["home_fix"] = dk["home_raw"].apply(fix_abbrev)

def get_opponent(row):
    if row["TeamFix"] == row["away_fix"]:
        return row["home_fix"]
    elif row["TeamFix"] == row["home_fix"]:
        return row["away_fix"]
    else:
        return None

dk["Opponent"] = dk.apply(get_opponent, axis=1)
dk["WeakOpponent"] = dk["Opponent"].isin(weak_teams).astype(int)

In [ ]:
# Step 3: Create adjusted projection
# ===================================================
# Hitters facing weak teams get +15%
# Pitchers facing weak teams get +10%

dk["AdjustedProj"] = dk["AvgPointsPerGame"].copy()

hitter_weak = (dk["WeakOpponent"] == 1) & (dk["is_hitter"])
pitcher_weak = (dk["WeakOpponent"] == 1) & (dk["is_pitcher"])

dk.loc[hitter_weak, "AdjustedProj"] = dk.loc[hitter_weak, "AvgPointsPerGame"] * 1.15
dk.loc[pitcher_weak, "AdjustedProj"] = dk.loc[pitcher_weak, "AvgPointsPerGame"] * 1.10

data["adjusted_proj"] = dict(zip(dk["ID"], dk["AdjustedProj"]))

## The Naive Baseline Model

Before testing my strategy, I need something to compare it against. That comparison point is called a naive model, this just means it makes no assumptions and uses no strategy. It simply asks: if you picked the 10 players with the highest average fantasy points per game, while staying under the salary cap and following all roster
rules, what lineup would you get?

This gives me a realistic baseline. If my weak opponent strategy cannot beat this simple approach, then my strategy was not worth it. If it does beat it, I have evidence that targeting weak opponents is a useful edge.

The crystal ball problem from Problem 1 is the ceiling, the best anyone could do on that day with perfect information. The naive model is the floor. The best someone could do with no strategy, just picking who looks the best on paper(Average Points per Game). My goal is to have my strategy in Problem 2, is to be above the floor.

In [ ]:
# Step 4: Build naive baseline model
# ================================================
# This model maximizes raw AvgPointsPerGame only.

DV_indexes = data["players"]
games = data["games"]
teams = data["teams"]

model_naive = pe.ConcreteModel()

model_naive.x = pe.Var(DV_indexes, domain=pe.Binary)
model_naive.y = pe.Var(games, domain=pe.Binary)

model_naive.obj = pe.Objective(
    expr=sum(data["proj"][i] * model_naive.x[i] for i in DV_indexes),
    sense=pe.maximize
)

# Salary cap
model_naive.salary_cap = pe.Constraint(
    expr=sum(data["salary"][i] * model_naive.x[i] for i in DV_indexes) <= 50000
)

# Exactly 10 players
model_naive.total_players = pe.Constraint(
    expr=sum(model_naive.x[i] for i in DV_indexes) == 10
)

# Position constraints
model_naive.twoP = pe.Constraint(expr=sum(model_naive.x[i] for i in data["P"]) == 2)
model_naive.oneC = pe.Constraint(expr=sum(model_naive.x[i] for i in data["C"]) == 1)
model_naive.one1B = pe.Constraint(expr=sum(model_naive.x[i] for i in data["1B"]) == 1)
model_naive.one2B = pe.Constraint(expr=sum(model_naive.x[i] for i in data["2B"]) == 1)
model_naive.one3B = pe.Constraint(expr=sum(model_naive.x[i] for i in data["3B"]) == 1)
model_naive.oneSS = pe.Constraint(expr=sum(model_naive.x[i] for i in data["SS"]) == 1)
model_naive.threeOF = pe.Constraint(expr=sum(model_naive.x[i] for i in data["OF"]) == 3)

# Max 5 hitters from one team
model_naive.max_hitters = pe.ConstraintList()
for t in teams:
    model_naive.max_hitters.add(
        sum(model_naive.x[i] for i in data["hitters_by_team"][t]) <= 5
    )

# At least 2 different games
model_naive.link_game = pe.ConstraintList()
for g in games:
    model_naive.link_game.add(
        sum(model_naive.x[i] for i in data["players_by_game"][g]) <= 10 * model_naive.y[g]
    )

model_naive.two_games = pe.Constraint(
    expr=sum(model_naive.y[g] for g in games) >= 2
)

# Solve naive model
opt = pe.SolverFactory("appsi_highs")
opt.solve(model_naive)

chosen_naive = [i for i in DV_indexes if pe.value(model_naive.x[i]) > 0.5]
lineup_naive = dk[dk["ID"].isin(chosen_naive)].copy()

lineup_naive = lineup_naive[[
    "Name", "TeamAbbrev", "Opponent", "Roster Position",
    "Salary", "AvgPointsPerGame", "ActualPoints"
]].sort_values("AvgPointsPerGame", ascending=False)


print("NAIVE BASELINE MODEL")
print("=" * 55)
print(f"Salary Used:             ${lineup_naive['Salary'].sum():,.0f}")
print(f"Total AvgPointsPerGame:  {lineup_naive['AvgPointsPerGame'].sum():.2f}")
print(f"Total Actual DK Points:  {lineup_naive['ActualPoints'].sum():.2f}")
display(lineup_naive)

NAIVE BASELINE MODEL
Salary Used:             $49,900
Total AvgPointsPerGame:  124.96
Total Actual DK Points:  62.00


,Name,TeamAbbrev,Opponent,Roster Position,Salary,AvgPointsPerGame,ActualPoints
39,Connelly Early,BOS,TB,P,8500,27.02,0.0
72,Trey Yesavage,TOR,KC,P,7700,24.25,0.0
142,Aaron Judge,NYY,BAL,OF,6600,11.68,3.0
343,Jakob Marsee,MIA,TEX,OF,4700,10.02,12.0
1020,Zach Cole,HOU,SEA,OF,2200,9.67,0.0
341,Luke Keaschall,MIN,CLE,2B,4800,9.52,7.0
325,Pete Alonso,NYM,WSH,1B,5000,8.95,12.0
405,Bo Bichette,TOR,KC,SS,4100,8.73,0.0
863,Isaac Paredes,HOU,SEA,3B,3200,8.03,5.0
891,Carter Jensen,KC,TOR,C,3100,7.09,23.0


## Strategy Model: Adjusted Projections
The naive model above uses raw average points per game with no adjustment. Now we run the same optimization model again, but this time using the adjusted projections that give a boost to players facing weak opponents. Everything else such as the salary cap and other constraints stays the exact same. The only thing that changes is the objective function because it uses `adjusted_proj` as the data.

In [ ]:
# Step 5: Build strategy model
# This model uses AdjustedProj instead of raw AvgPointsPerGame.

model_2 = pe.ConcreteModel()

model_2.x = pe.Var(DV_indexes, domain=pe.Binary)
model_2.y = pe.Var(games, domain=pe.Binary)

model_2.obj = pe.Objective(
    expr=sum(data["adjusted_proj"][i] * model_2.x[i] for i in DV_indexes),
    sense=pe.maximize
)

# Salary cap
model_2.salary_cap = pe.Constraint(
    expr=sum(data["salary"][i] * model_2.x[i] for i in DV_indexes) <= 50000
)

# Exactly 10 players
model_2.total_players = pe.Constraint(
    expr=sum(model_2.x[i] for i in DV_indexes) == 10
)

# Position constraints
model_2.twoP = pe.Constraint(expr=sum(model_2.x[i] for i in data["P"]) == 2)
model_2.oneC = pe.Constraint(expr=sum(model_2.x[i] for i in data["C"]) == 1)
model_2.one1B = pe.Constraint(expr=sum(model_2.x[i] for i in data["1B"]) == 1)
model_2.one2B = pe.Constraint(expr=sum(model_2.x[i] for i in data["2B"]) == 1)
model_2.one3B = pe.Constraint(expr=sum(model_2.x[i] for i in data["3B"]) == 1)
model_2.oneSS = pe.Constraint(expr=sum(model_2.x[i] for i in data["SS"]) == 1)
model_2.threeOF = pe.Constraint(expr=sum(model_2.x[i] for i in data["OF"]) == 3)

# Max 5 hitters from one team
model_2.max_hitters = pe.ConstraintList()
for t in teams:
    model_2.max_hitters.add(
        sum(model_2.x[i] for i in data["hitters_by_team"][t]) <= 5
    )

# At least 2 different games
model_2.link_game = pe.ConstraintList()
for g in games:
    model_2.link_game.add(
        sum(model_2.x[i] for i in data["players_by_game"][g]) <= 10 * model_2.y[g]
    )

model_2.two_games = pe.Constraint(
    expr=sum(model_2.y[g] for g in games) >= 2
)

# Solve strategy model
opt.solve(model_2)

chosen_id_2 = [i for i in DV_indexes if pe.value(model_2.x[i]) > 0.5]
lineup2 = dk[dk["ID"].isin(chosen_id_2)].copy()

lineup2 = lineup2[[
    "Name", "TeamAbbrev", "Opponent", "WeakOpponent", "Roster Position",
    "Salary", "AvgPointsPerGame", "AdjustedProj", "ActualPoints"
]].sort_values("AdjustedProj", ascending=False)


print("PROBLEM 2 RESULTS: WEAK OPPONENT STRATEGY")
print("=" * 55)
print(f"Salary Used:              ${lineup2['Salary'].sum():,.0f}")
print(f"Total AvgPointsPerGame:   {lineup2['AvgPointsPerGame'].sum():.2f}")
print(f"Total Adjusted Proj:      {lineup2['AdjustedProj'].sum():.2f}")
print(f"Total Actual DK Points:   {lineup2['ActualPoints'].sum():.2f}")
display(lineup2)

PROBLEM 2 RESULTS: WEAK OPPONENT STRATEGY
Salary Used:              $49,900
Total AvgPointsPerGame:   123.13
Total Adjusted Proj:      128.32
Total Actual DK Points:   79.00


,Name,TeamAbbrev,Opponent,WeakOpponent,Roster Position,Salary,AvgPointsPerGame,AdjustedProj,ActualPoints
39,Connelly Early,BOS,TB,0,P,8500,27.02,27.0200,0.0
72,Trey Yesavage,TOR,KC,0,P,7700,24.25,24.2500,0.0
213,Juan Soto,NYM,WSH,1,OF,6200,10.58,12.1670,32.0
325,Pete Alonso,NYM,WSH,1,1B,5000,8.95,10.2925,12.0
358,Zach Neto,LAA,COL,1,SS,4500,8.91,10.2465,0.0
343,Jakob Marsee,MIA,TEX,0,OF,4700,10.02,10.0200,12.0
1020,Zach Cole,HOU,SEA,0,OF,2200,9.67,9.6700,0.0
341,Luke Keaschall,MIN,CLE,0,2B,4800,9.52,9.5200,7.0
863,Isaac Paredes,HOU,SEA,0,3B,3200,8.03,8.0300,5.0
881,Francisco Alvarez,NYM,WSH,1,C,3100,6.18,7.1070,11.0


## Results Summary

Total Actual Points: 79.00  

This strategy underperformed significantly compared to the optimization model (268.15 points), showing that simple heuristics are not effective under multiple constraints.

In [ ]:
# Step 6: Constraint checks
# =============================================
print("\n--- Constraint Checks ---")
print("Players selected:", len(lineup2))
print("Pitchers selected:", lineup2["Roster Position"].str.contains(r"\bP\b", regex=True).sum())
print("Unique games:", dk[dk["ID"].isin(chosen_id_2)]["gamePk"].nunique())

hitters2 = lineup2[~lineup2["Roster Position"].str.contains(r"\bP\b", regex=True)]
print("Max hitters from one team:", hitters2.groupby("TeamAbbrev")["Name"].count().max())


--- Constraint Checks ---
Players selected: 10
Pitchers selected: 2
Unique games: 7
Max hitters from one team: 3


In [ ]:
# Step 7: Final comparison
# ================================================
naive_actual = lineup_naive["ActualPoints"].sum()
strategy_actual = lineup2["ActualPoints"].sum()

crystal_actual = lineup1["ActualPoints"].sum()


print("FINAL COMPARISON")
print("=" * 55)
print(f"{'Model':<30} {'Actual DK Pts':>14}")
print("-" * 45)
print(f"{'Crystal Ball (Problem 1)':<30} {crystal_actual:>14.2f}")
print(f"{'Weak Opponent Strategy':<30} {strategy_actual:>14.2f}")
print(f"{'Naive Baseline (raw avg)':<30} {naive_actual:>14.2f}")

print()
print(f"Strategy vs. Naive: {strategy_actual - naive_actual:+.2f} pts")
print(f"Gap to Crystal Ball: {crystal_actual - strategy_actual:.2f} pts")

FINAL COMPARISON
Model                           Actual DK Pts
---------------------------------------------
Crystal Ball (Problem 1)               268.15
Weak Opponent Strategy                  79.00
Naive Baseline (raw avg)                62.00

Strategy vs. Naive: +17.00 pts
Gap to Crystal Ball: 189.15 pts


## Project Summary

This project used two linear programming models to build two DraftKings MLB linup optimization models for September 19, 2025

**Problem 1- Crystal Ball Lineup:** By giving the model each player's actual performance from that day, the solver found the perfect lineup for that day, making sure it met all of the constraints and DraftKings MLB Classic fantasy sports ruleset. This gave me what the best anyone could have done with perfect information. The result is I got 268.15 which meant that was the highest amount of DraftKings points a lineup could receive from that day while following all of the rules.

**Problem 2 — Weak Opponent Strategy:** Without knowing how players would perform that day like in Problem 1, I tested whether boosting players who faced bad teams(win percentage under .450) could produce a better lineup than just picking the highest average scorers who meet the salary. I compared three lineups: the crystal ball ceiling, a naive average-based lineup, and our adjusted strategy lineup. My weak opponent strategy performed 17 points better than the Naive model.

**What I learned:** The gap between the crystal ball and any real-world strategy shows how much uncertainty costs in daily fantasy sports. Even though my real world strategy did better than just picking the most optimal lineup based on Average Points per Game, it still came up way short from the 268.15 ceiling the crystal ball model set. Fantasy sports and gambling are unpredictable. You might think you have the perfect strategy, but most of the time your strategy comes up well short of the actual amount. This is something I have faced in this project, where you think you are better than the average person playing this game, only to put up a measly 79 points, which is a 189.15 points short from the best result. The big picture is if you're playing fantasy sports games for money, assume you are going to lose what you are putting in. Never use fantasy sports or gambling as something you do when you need money.

## AI Appendix

**How I used AI:**
- I used Claude throughout this project mainly as a debugging partner. I asked it questions when I ran into errors or when my model was producing unexpected results, and I used its suggestions to help diagnose issues in the code.

**What it helped with:**
- Prompt: How can I make this loading of data into something more optimal as a function that I can call and make it quicker for each time I load?
Context: I wrote a load in for the data and JSON, but I was wondering if I could do something quicker since I know I am going to be working on this project over a longer period of time. I asked AI to change my loading and dataframe from the project in the document into a function I could call to make it easier for future uses in this document. The output was it made my first draft work without a function into the `load_and_setup` function.
- It helped with the `abbrev_fix` function, I didn't know all the abbreviations that MLB API and DraftKings could store a team as, so I decided to hardcode it but AI helped fill in what all possible abbreviations for each team could be.

- The == equality constraints on positions made it mathematically impossible to select any 2B/OF player like Massey. Switching to >= lower bounds + <= upper bounds, with total_players == 10 enforcing the exact count, correctly reflects how DraftKings flex eligibility actually works.
- Suggesting the `unicodedata` fix for accent marks on Latin players' names  
- Another issue AI helped fix was with my merge. My original code ran two separate merges to assign game IDs to players, but the second merge was overwriting the abbreviation fixes from the first. Claude caught this and suggested dropping the stale column before merging on the normalized matchup key.
- I also used GAI to clean up my grammar for my markdowns.

**Where AI did not help:**  
AI could not run my code or access my actual data, so every suggestion had to be tested and verified by me. Sometimes fixes that seemed correct still produced errors in my notebook, which meant I had to experiment and adjust the solutions myself.

**What I learned:**  
I learned how to structure a large optimization workflow, including pulling data from an API, cleaning and merging datasets. This was a very tough project and I am glad I started this project in mid February. GAI while it was very helful in debugging my code, would often times suggest things I never learned. I remember when it was debugging my second problem because I messed up the list comprehension in `chosen_id2`, it kept saying how it could improve my problem and give me a higher output because I pasted in the whole block. I declined it because I knew I couldn't explain what it gave or know if it was even right or wrong.